# EDA の進め方

目的変数の理解から欠損・型・相関の確認、データリークの兆候発見まで、EDA の流れを体系的に学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/python-data-analysis/06-eda`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# Kaggle Notebook 上では既存の /kaggle/input/titanic をそのまま使用し、
# Google Colab やローカル環境では互換スキーマの Titanic サンプルデータ（891行 / 418行）を自動配置します。
# ※ Kaggle 本番データで提出する場合は kaggle CLI (!kaggle competitions download -c titanic) をご利用ください。
import os
import numpy as np
import pandas as pd

def _ensure_titanic_data():
    target_dir = "/kaggle/input/titanic"
    if os.path.exists(os.path.join(target_dir, "train.csv")):
        return

    try:
        os.makedirs(target_dir, exist_ok=True)
        data_dir = target_dir
    except OSError:
        data_dir = os.path.abspath("./.kaggle_input/titanic")
        os.makedirs(data_dir, exist_ok=True)
        _orig_read_csv = pd.read_csv

        def _compat_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/"):
                rel = filepath_or_buffer[len("/kaggle/input/"):]
                local_path = os.path.abspath(os.path.join("./.kaggle_input", rel))
                if os.path.exists(local_path):
                    filepath_or_buffer = local_path
            return _orig_read_csv(filepath_or_buffer, *args, **kwargs)

        pd.read_csv = _compat_read_csv

    rng = np.random.default_rng(42)
    surnames = ["Braund", "Cumings", "Heikkinen", "Futrelle", "Allen", "Moran", "McCarthy", "Palsson", "Johnson", "Nasser"]
    given_m = ["Owen", "William", "James", "Henry", "Leonard", "Timothy", "charles", "George"]
    given_f = ["Florence", "Laina", "Lily", "elisabeth", "Adele", " Marguerite", "Anna", "Mary"]

    def _make_split(n, start_id, with_target):
        pclass = rng.choice([1, 2, 3], size=n, p=[0.24, 0.21, 0.55])
        sex = rng.choice(["male", "female"], size=n, p=[0.65, 0.35])
        age = np.clip(rng.normal(29.7, 14.5, size=n), 0.42, 80.0).round(1)
        sibsp = rng.choice([0, 1, 2, 3, 4], size=n, p=[0.68, 0.23, 0.05, 0.02, 0.02])
        parch = rng.choice([0, 1, 2, 3], size=n, p=[0.76, 0.13, 0.09, 0.02])
        base_fare = np.where(pclass == 1, 60.0, np.where(pclass == 2, 20.0, 11.0))
        fare = (base_fare * rng.lognormal(0.0, 0.55, size=n)).round(2)
        embarked = rng.choice(["S", "C", "Q"], size=n, p=[0.72, 0.19, 0.09]).astype(object)

        names = []
        cabins = []
        tickets = []
        for i in range(n):
            sn = surnames[i % len(surnames)]
            if sex[i] == "male":
                title = "Master" if age[i] < 13 else "Mr"
                gn = given_m[i % len(given_m)].strip().capitalize()
            else:
                title = "Miss" if age[i] < 24 else "Mrs"
                gn = given_f[i % len(given_f)].strip().capitalize()
            names.append(f"{sn}, {title}. {gn}")
            cabins.append(f"{'ABC'[pclass[i] - 1]}{10 + (i % 90)}" if rng.random() > 0.77 else np.nan)
            tickets.append(f"PC {17500 + (i % 350)}" if pclass[i] == 1 else str(230000 + i))

        age_obj = age.astype(float)
        age_obj[rng.random(n) < 0.20] = np.nan
        embarked[rng.random(n) < 0.005] = np.nan
        if not with_target:
            fare[0] = np.nan

        data = {
            "PassengerId": np.arange(start_id, start_id + n),
            "Pclass": pclass,
            "Name": names,
            "Sex": sex,
            "Age": age_obj,
            "SibSp": sibsp,
            "Parch": parch,
            "Ticket": tickets,
            "Fare": fare,
            "Cabin": pd.Series(cabins, dtype=object),
            "Embarked": pd.Series(embarked, dtype=object),
        }
        if with_target:
            family_size = sibsp + parch + 1
            logit = (
                -1.65
                + np.where(sex == "female", 2.45, 0.0)
                + np.where(pclass == 1, 1.35, np.where(pclass == 2, 0.35, -0.55))
                + np.where(np.nan_to_num(age_obj, nan=29.0) < 12, 1.2, 0.0)
                + np.where((family_size >= 2) & (family_size <= 4), 0.75, -0.35)
            )
            prob = 1.0 / (1.0 + np.exp(-logit))
            survived = (rng.random(n) < prob).astype(int)
            ordered = {"PassengerId": data["PassengerId"], "Survived": survived}
            for k, v in data.items():
                if k != "PassengerId":
                    ordered[k] = v
            return pd.DataFrame(ordered)
        return pd.DataFrame(data)

    _make_split(891, 1, True).to_csv(os.path.join(data_dir, "train.csv"), index=False)
    _make_split(418, 892, False).to_csv(os.path.join(data_dir, "test.csv"), index=False)
    print(f"Titanic dataset ready at: {data_dir}")

_ensure_titanic_data()

## EDA とは

EDA（探索的データ分析）は、モデルを作る前にデータを観察し、
**前処理・特徴量・検証設計の方針を立てる**工程です。
やみくもにグラフを描くのではなく、目的を持って順に確認していきます。

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/titanic/train.csv")

## 1. 目的変数を理解する

まず予測対象（目的変数）の性質を把握します。分類なら**クラスの偏り（不均衡）**が重要です。

In [ ]:
df["Survived"].value_counts(normalize=True)   # クラス比率

クラスが偏っていると、正解率（accuracy）では性能を見誤ります。評価指標の選択にも影響します。

## 2. 欠損・型・基本統計を確認する

In [ ]:
df.info()                       # 型と非欠損数
df.isnull().mean().sort_values(ascending=False)  # 列ごとの欠損率
df.describe(include="all")      # 数値・カテゴリ両方の要約

型が想定と違う列（数値が文字列になっている等）や、極端な最大・最小値をここで見つけます。

## 3. 特徴量と目的変数の関係を見る

In [ ]:
# 数値特徴：目的変数で層別した分布
df.groupby("Survived")["Fare"].median()

# カテゴリ特徴：水準ごとの目的変数平均
df.groupby("Pclass")["Survived"].mean()

「効きそうな特徴」の当たりをつけ、次トラックの特徴量エンジニアリングへつなげます。

## 4. データリークの兆候に気づく

> **Warning:** **単一の特徴だけで目的変数をほぼ完璧に説明できる**場合、データリークを疑います。 「本来は予測時点で手に入らない情報」が紛れ込むと、検証スコアは高いのに本番で崩れます。 ID 列・時刻・集計済みの値などは要注意です。

リークを見抜くには、「その特徴は予測したい瞬間に実際に存在するか？」を一つずつ問うのが有効です。

## EDA から検証設計へ

EDA で分かったデータの構造（時系列性・グループ構造・不均衡）は、
そのまま交差検証の分け方に反映します。
データの性質に合わない検証は、見かけのスコアを当てにできなくします。

> **Tip:** EDA の発見は Markdown セルにメモしておきましょう。「なぜこの前処理をするのか」の根拠になり、 後から自分やチームが方針を追えるようになります。

## 次のステップ

次は、EDA で見つかる代表的な課題である**欠損値・外れ値**の検出と対処方針を詳しく学びます。